# SofaScore Overall Kickbase-Points Approximation

This notebook estimates post-match Kickbase points from SofaScore's lineups, incidents, and shot-map endpoints. It uses only each team's `overall_matches` window from the latest team-form snapshot; Bundesliga-only points remain sourced from Kickbase. Every calculated match total and its individual awards are retained in the JSON output.

Direct SofaScore actions and approved proxies are labelled separately in the exported award ledger. Unsupported actions are deliberately not estimated.

In [1]:
from __future__ import annotations

import importlib
import json
import sys
from pathlib import Path


def locate_project_root() -> Path:
    starts = []
    notebook_path = globals().get('__vsc_ipynb_file__')
    if isinstance(notebook_path, str) and notebook_path.strip():
        starts.append(Path(notebook_path).expanduser().resolve().parent)
    starts.append(Path.cwd().resolve())
    for start in starts:
        for candidate in (start, *start.parents):
            if (candidate / 'project_paths.py').is_file():
                return candidate
    raise FileNotFoundError('Could not locate project_paths.py. Start Jupyter from the project root.')


PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from project_paths import KICKBASE_REFERENCE_DIR
import sofascore_kickbase_points as kickbase_points

# Reload lets a rerun use local scoring-engine edits without restarting Jupyter.
kickbase_points = importlib.reload(kickbase_points)

catalog = kickbase_points.MetricCatalog.from_document(json.loads((KICKBASE_REFERENCE_DIR / 'kickbase_metrics.json').read_text(encoding='utf-8')))
kickbase_points.validate_scoring_contract(catalog)
print('Scoring contract validated: Big Chance Created = +15; woodwork variants = +10 once.')
print(json.dumps(kickbase_points.scoring_policy(), indent=2))

Scoring contract validated: Big Chance Created = +15; woodwork variants = +10 once.
{
  "big_chance_created_points": 15,
  "ignored_metric_ids": [
    "big_chance_zero",
    "secondary_assist",
    "own_goal_forced",
    "deflected_assist",
    "deadly_pass",
    "rebound_assist",
    "woodwork_assist",
    "dive_save",
    "last_man_tackle",
    "challenged_collection",
    "keeper_sweeper",
    "dive_catch",
    "standing_saved",
    "unchallenged_collection",
    "corner_won",
    "cross_blocked",
    "accurate_throw",
    "cross_block_possession",
    "cross_not_claimed",
    "incorrect_throw_in"
  ],
  "woodwork_group": {
    "metric_ids": [
      "post_label_crossbar",
      "left_post",
      "right_post"
    ],
    "awarded_as": "post_label_crossbar",
    "points": 10
  },
  "confidence_labels": {
    "direct": "direct SofaScore field or event",
    "derived": "reconstructed from multiple SofaScore payloads",
    "proxy": "aggregate or location-based approximation",
    "condit

## Confirmed validation rules

The collector validates these confirmed mappings before fetching data:

- `inGamePenalty` + `reason: goalkeeperSave` credits a normal-time goalkeeper penalty save.
- `penaltyShootout` `scored`/`missed` events and `reason: goalkeeperSave` handle shoot-out attempts.
- `statistics.clearanceOffLine` and `statistics.errorLeadToAShot` are direct awards.
- `statistics.bigChanceCreated` is worth +15.
- Any supported woodwork shot type is awarded as one +10 woodwork event.

The export records source and confidence on each award so proxy calculations can be audited.

In [2]:
# Set True only when Chrome should run without a visible browser window.
JSON_OUTPUT_PATH, CSV_OUTPUT_PATH = kickbase_points.run_notebook_workflow(headless=False)

Current Bundesliga matchday (1-34):  2


Using team-form input: team_form_2026-09-01_14-41-46_881236+0200.json
Loaded 18 teams; evaluating overall matches only.
[1/18] FC Bayern München (team_id=2672)
[2/18] VfB Stuttgart (team_id=2677)
[3/18] 1. FC Köln (team_id=2671)
[4/18] TSG Hoffenheim (team_id=2569)
[5/18] 1. FC Union Berlin (team_id=2547)
[6/18] Eintracht Frankfurt (team_id=2674)
[7/18] 1. FSV Mainz 05 (team_id=2556)
[8/18] SC Paderborn 07 (team_id=2561)
[9/18] RB Leipzig (team_id=36360)
[10/18] Borussia M'gladbach (team_id=2527)
[11/18] SV 07 Elversberg (team_id=2598)
[12/18] Bayer 04 Leverkusen (team_id=2681)
[13/18] Borussia Dortmund (team_id=2673)
[14/18] Hamburger SV (team_id=2676)
[15/18] SC Freiburg (team_id=2538)
[16/18] SV Werder Bremen (team_id=2534)
[17/18] FC Augsburg (team_id=2600)
[18/18] FC Schalke 04 (team_id=2530)
Unique match IDs: 77
Successful SofaScore requests: 231; failed matches: 0; cache hits: 13
JSON output: C:\kickbase project\outputs\sofascore\player_kickbase_point_averages\overall_player_kic